# 01. SyntheticMetalSeg 데이터셋 목표와 설계

1장의 목적은 조명, 각도, 반사, 카메라 설정을 통제할 수 있는 공용 synthetic segmentation benchmark를 만드는 것입니다.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "synthetic_metal_utils.py").exists():
    search_roots = [Path.cwd(), *Path.cwd().parents]
    search_patterns = ["synthetic_metal_utils.py", "*/synthetic_metal_utils.py", "*/*/synthetic_metal_utils.py"]
    for root in search_roots:
        for pattern in search_patterns:
            matches = list(root.glob(pattern))
            if matches:
                NOTEBOOK_DIR = matches[0].parent
                break
        if (NOTEBOOK_DIR / "synthetic_metal_utils.py").exists():
            break

sys.path.append(str(NOTEBOOK_DIR))
DATA_ROOT = NOTEBOOK_DIR / "data" / "synthetic_metal_seg"

from synthetic_metal_utils import *
set_korean_font()
set_seed(7)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("DATA_ROOT:", DATA_ROOT)

## 설계 목표

데이터셋은 단순한 toy image가 아니라 일반화 실패를 재현하기 위한 실험 장치입니다.

In [ ]:
config = default_generation_config()
config

## 공용 산출물

이후 장은 모두 같은 DATA_ROOT와 metadata 규격을 사용합니다.

In [ ]:
expected = {
    "images": ["train", "val_seen", "val_unseen_light", "val_unseen_angle", "val_unseen_material", "val_worst_combo"],
    "masks": ["train", "val_seen", "val_unseen_light", "val_unseen_angle", "val_unseen_material", "val_worst_combo"],
    "metadata": ["samples.csv", "domains.json", "split_config.json", "generation_config.json"],
    "previews": ["sample_grid.png", "mask_overlay_grid.png", "domain_color_histograms.png", "split_summary.png"],
}
expected

## 실제 검사 환경과 대응

synthetic parameter가 실제 설비 변수와 대응되어야 실험 결과를 현장으로 되돌릴 수 있습니다.

In [ ]:
mapping = {
    "light_r/g/b": "RGB 조명 채널 intensity",
    "light_angle": "조명 조사 방향 또는 조명 위치",
    "camera_angle": "카메라/제품 자세 편차",
    "metal_roughness": "표면 재질과 반사 편차",
    "exposure/gamma": "카메라 gain 또는 image processing 설정",
    "glare_strength": "금속 표면 highlight 강도",
    "domain_id": "조명/제품/촬영 조건 조합",
}
mapping